# 01_load_inspect

**Input:** 12 monthly CSV files in `data/raw/` (not in the repo, see the README for where to download them)

**Output:** one combined file, `rides_combined_raw.parquet`, and a log of rows per file in `reports/load_log.csv`

**Why it exists:** Loads and combines the monthly files and checks them (null, duplicates, ride lengths) without changing the data, so the cleaning rules in notebook 02 are based on what was found.

In [1]:
# Read all raw CSVs with fixed column types, tag each row with its source file, combine and save as parquet, and write the rows-per-file log to load_log.csv

import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/clean")
files = sorted(RAW.glob("*.csv"))
assert len(files) == 12, f"expected 12 files, found {len(files)}"

str_cols = ["ride_id", "rideable_type", "start_station_name", "start_station_id",
            "end_station_name", "end_station_id", "member_casual"]
float_cols = ["start_lat", "start_lng", "end_lat", "end_lng"]
dtypes = {**{c: "string" for c in str_cols}, **{c: "float32" for c in float_cols}}

frames, load_log = [], []
for f in files:
    df = pd.read_csv(f, dtype=dtypes)
    df["started_at"] = pd.to_datetime(df["started_at"], format="ISO8601")
    df["ended_at"] = pd.to_datetime(df["ended_at"], format="ISO8601")
    df["source_file"] = f.name
    load_log.append({"file": f.name, "rows": len(df)})
    frames.append(df)

rides = pd.concat(frames, ignore_index=True)
del frames

for c in ["rideable_type", "member_casual", "start_station_name", "start_station_id",
          "end_station_name", "end_station_id", "source_file"]:
    rides[c] = rides[c].astype("category")

print(pd.DataFrame(load_log))
print("TOTAL:", len(rides))
rides.to_parquet(OUT / "rides_combined_raw.parquet")
pd.DataFrame(load_log).to_csv("../reports/load_log.csv", index=False)

                         file    rows
0   202510-divvy-tripdata.csv  646039
1   202511-divvy-tripdata.csv  356628
2   202512-divvy-tripdata.csv  140534
3   202601-divvy-tripdata.csv  137787
4   202602-divvy-tripdata.csv  201450
5   202603-divvy-tripdata.csv  317037
6   202604-divvy-tripdata.csv  448252
7   202605-divvy-tripdata.csv  653704
8   202606-divvy-tripdata.csv  762550
9   202607-divvy-tripdata.csv  869051
10  202608-divvy-tripdata.csv  868191
11  202609-divvy-tripdata.csv  770562
TOTAL: 6171785


In [3]:
# First look at the combined data: missing values per column, date range, rider and bike mix, duplicate ride IDs, ride-length extremes

rides.info(memory_usage="deep")
print(rides.isna().sum())
print(rides[["started_at", "ended_at"]].agg(["min", "max"]))
print(rides["member_casual"].value_counts(dropna=False))
print(rides["rideable_type"].value_counts(dropna=False))
print("duplicate ride_ids:", rides["ride_id"].duplicated().sum())

dur = (rides["ended_at"] - rides["started_at"]).dt.total_seconds()
print(dur.describe())
print("negative:", (dur < 0).sum(), "| zero:", (dur == 0).sum(),
      "| over 24h:", (dur > 86400).sum())

<class 'pandas.DataFrame'>
RangeIndex: 6171785 entries, 0 to 6171784
Data columns (total 14 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ride_id             string        
 1   rideable_type       category      
 2   started_at          datetime64[us]
 3   ended_at            datetime64[us]
 4   start_station_name  category      
 5   start_station_id    category      
 6   end_station_name    category      
 7   end_station_id      category      
 8   start_lat           float32       
 9   start_lng           float32       
 10  end_lat             float32       
 11  end_lng             float32       
 12  member_casual       category      
 13  source_file         category      
dtypes: category(7), datetime64[us](2), float32(4), string(1)
memory usage: 394.6 MB
ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1289486
start_station_id      1

In [4]:
# Check whether each file holds only rides that started in its own month, and what happened to those that didn't

rides["file_month"] = pd.to_datetime(rides["source_file"].str[:6], format="%Y%m")
early = rides[rides["started_at"] < rides["file_month"]]
print("started before file month:", len(early))
print("of those, ended in file month:",
      (early["ended_at"].dt.to_period("M") == early["file_month"].dt.to_period("M")).mean())

started before file month: 734
of those, ended in file month: 1.0
